# Treinamento

As colunas da database são: PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked

### Pré processamneto realizado
    As colunas 'PassengerId',"Name",'Ticket',"Cabin" foram cortadas por serem consideradas ruído ou pouco significativas para o resultado.
    A coluna 'Sex' foi transformada em valor binário
    A coluna 'Embarked' foi transformada em 3 colunas binárias cada uma referente a um dos valores possiveis de onde a pessoa embarcou.

### Modelo usado
    O xgboost foi usado como modelo caixa preta que cria uma série de arvores de decisão (Tradicionalmente considerados caixa branca) e usa o resultado dessas arvores para tomar uma decisão, de modo que a sáida é matematicamente complexa o suficiente para ser humamente dificil explicar.

### Hiperparametros
    A taxa de aprendizado é de 0.1 e a profundidade da árvore é de 5. Outras combinações de valores foram tentadas mas essa atingiu o maior resultado em termos de acurácia (0.8182).
    _Obs_: Usando os parametros padrão do xgboost a acurácia foi de 0.7622

### Valores alterados
    Houveram tentativas de ajuste menores, como cortar outras colunas, mas todas as combinações e variações diferentes das que estão na célula abaixo apresentaram redução na acurácia.
    

In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

dataset = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(dataset)

df = df.drop(['PassengerId',"Name",'Ticket',"Cabin"],axis=1).dropna()

le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
df = pd.get_dummies(df, columns=['Embarked'])

X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBClassifier(learning_rate=0.1,max_depth=5)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Acurácia: {accuracy:.4f}")

Acurácia: 0.8182


In [45]:
import shap
import numpy as np

instancia = X_test.iloc[0:1]
background = shap.sample(X_train, 10)

def predict_fn(X):
    return model.predict_proba(X)

explainer = shap.KernelExplainer(predict_fn, background)

nsamples_lista = [10, 15, 20, 25, 30]
explicacoes_pobres = []

for n in nsamples_lista:
    shap_values = explainer.shap_values(instancia, nsamples=n, silent=True)
    valores_brutos = np.array(shap_values)
    if valores_brutos.ndim == 3:
        valores = valores_brutos[0, :, 1]
    elif isinstance(shap_values, list):
        valores = shap_values[1][0]
    else:
        valores = valores_brutos[0]
        
    exp_formatada = [(X_train.columns[i], float(valores[i])) for i in range(len(X_train.columns))]
    explicacoes_pobres.append(exp_formatada)

for idx, exp in enumerate(explicacoes_pobres):
    print(f"--- Explicação {idx+1} (nsamples={nsamples_lista[idx]}) ---")
    exp_ordenada = sorted(exp, key=lambda x: abs(x[1]), reverse=True)
    for feature, valor in exp_ordenada:
        print(f"  {feature}: {valor:.4f}")
    print()

--- Explicação 1 (nsamples=10) ---
  Sex: 0.4043
  Pclass: 0.0687
  Parch: -0.0664
  Age: -0.0113
  Fare: 0.0059
  Embarked_C: 0.0011
  SibSp: 0.0000
  Embarked_Q: 0.0000
  Embarked_S: 0.0000

--- Explicação 2 (nsamples=15) ---
  Sex: 0.4184
  Age: -0.1201
  SibSp: 0.0761
  Pclass: 0.0694
  Fare: -0.0482
  Embarked_S: 0.0146
  Parch: -0.0141
  Embarked_C: 0.0063
  Embarked_Q: 0.0000

--- Explicação 3 (nsamples=20) ---
  Sex: 0.3758
  Age: -0.0855
  SibSp: 0.0715
  Pclass: 0.0469
  Embarked_S: -0.0314
  Fare: 0.0310
  Parch: -0.0050
  Embarked_C: -0.0009
  Embarked_Q: 0.0000

--- Explicação 4 (nsamples=25) ---
  Sex: 0.3345
  Pclass: 0.0774
  Age: -0.0316
  Embarked_C: 0.0119
  Fare: 0.0102
  Parch: 0.0019
  Embarked_S: -0.0017
  SibSp: -0.0003
  Embarked_Q: 0.0000

--- Explicação 5 (nsamples=30) ---
  Sex: 0.3329
  Pclass: 0.0731
  Fare: 0.0298
  Age: -0.0229
  Parch: -0.0177
  Embarked_S: 0.0051
  SibSp: 0.0045
  Embarked_C: -0.0025
  Embarked_Q: 0.0000

